### **Import modules and setting the env vars**

In [1]:
import imaplib, email, ssl, os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd().parent / "private" / ".env"
load_dotenv(env_path)

imap_server = os.environ["IITB_IMAP_SERVER"]
email_address = os.environ["IITB_EMAIL"]
access_token = os.environ["IITB_IMAP_ACCESS_TOKEN"]

In [2]:
# 1. Create a modern default SSL context
context = ssl.create_default_context()

# 2. Lower the restriction to allow OpenSSL's standard 'DEFAULT' ciphers
# This bridges the gap with servers requiring legacy key exchanges.
# IITB IMAP server need this
context.set_ciphers('DEFAULT')

context

In [3]:
try:
    print(f"Connecting securely to {imap_server}...")

    # 3. Pass the custom context into IMAP4_SSL
    mail = imaplib.IMAP4_SSL(imap_server, 993, ssl_context=context)
    mail.login(email_address, access_token)
    print("Login successful!")
    mail.logout()

except ssl.SSLError as ssl_err:
    print(f"SSL Specific Error: {ssl_err}")
except Exception as e:
    print(f"Connection failed: {e}")

Connecting securely to imap.iitb.ac.in...
Login successful!


### **Now We check getting all the emails from the server**

In [4]:
try:
    print(f"Connecting securely to {imap_server}...")
    mail = imaplib.IMAP4_SSL(imap_server, 993, ssl_context=context)
    mail.login(email_address, access_token)
    print("Login successful!")

    mail.select("INBOX", readonly=True)
    status, messages = mail.search(None, "ALL")
    for num in messages[0].split()[::-1]:
        _, data = mail.fetch(num, "(BODY.PEEK[])")
        msg = email.message_from_bytes(data[0][1])
        print(25*"---")
        print(msg["Subject"])
        print(msg["From"])
        print(msg["Date"])

    mail.logout()

except ssl.SSLError as ssl_err:
    print(f"SSL Specific Error: {ssl_err}")
except Exception as e:
    print(f"Connection failed: {e}")

Connecting securely to imap.iitb.ac.in...
Login successful!
---------------------------------------------------------------------------
[S.Events] Indian Strategy and Trading Conclave 2027 | Quant Club
General Secretary Technical Affairs <gsectech@iitb.ac.in>
Mon, 28 Sep 2026 23:44:58 +0530
---------------------------------------------------------------------------
CS 740-2026-1: Course projects
"Ajit Rajwade (via IITB Moodle)" <noreply@moodle.iitb.ac.in>
Mon, 28 Sep 2026 19:11:03 +0530
---------------------------------------------------------------------------
=?UTF-8?Q?Calling_all_Hostel_15_PG_Residents=3A_Let=27s_Unite_an?=
 =?UTF-8?Q?d_Conquer_Gyrations_GC!_=F0=9F=8F=86?=
Cultural Council H2 <cultcoh2@iitb.ac.in>
Mon, 28 Sep 2026 18:57:57 +0530
---------------------------------------------------------------------------
[Student-notices] Invitation to Participate in Cleanliness Drive on 3rd
 October 2026 at 7.30 a.m.
Public Health Officer <phoiitb@iitb.ac.in>
Mon, 28 Sep 2026 18:51:

### **Now We extract event details in json format using free langchain api**

In [5]:
prompt = """Extract title, venue, date, time, organizer and audience from the notice below.
Audience must be exactly one of: Public, Institute-wide, Department-specific, Club-specific, Personal.
Return only valid JSON with those keys."""

In [6]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0", task="text-generation", temperature=0.01)
model = ChatHuggingFace(llm=llm)
result = model.invoke("what is the capital of India?") # replace prompt here

print(result.content)

The capital of India is New Delhi.
